### Cell 1 — Setup

In [0]:
# ============================================================
#  04b_ML_FORECASTING — TOURGO ML PIPELINE
#  Day 7: Revenue Forecasting với Linear Regression
#  Input : gold_revenue_daily
#  Output: ml_revenue_forecast
# ============================================================

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import (
    col, dayofweek, month, lag,
    avg as _avg, lit
)
from pyspark.sql.window import Window
import mlflow

spark.sql("USE tourgo")
mlflow.set_experiment("/Shared/tourgo-ml-experiments")

df_rev = spark.read.table("gold_revenue_daily").orderBy("booking_date")
total_days = df_rev.count()

print(f"-> gold_revenue_daily: {total_days} ngày")
print(f"   Cần ít nhất 10 ngày để train/test split")
if total_days < 10:
    print("=>  Dữ liệu ít — R² có thể thấp, ghi chú trong slide")
df_rev.show(5, truncate=False)

-> gold_revenue_daily: 32 ngày
   Cần ít nhất 10 ngày để train/test split
+------------+-------------+----------------+------------+----------------+-------+
|booking_date|total_revenue|provider_revenue|platform_fee|num_transactions|month  |
+------------+-------------+----------------+------------+----------------+-------+
|2026-04-22  |1.57E7       |1.413E7         |1570000.0   |3               |2026-04|
|2026-04-23  |3.81E7       |3.429E7         |3810000.0   |6               |2026-04|
|2026-04-25  |3.43E7       |3.087E7         |3430000.0   |5               |2026-04|
|2026-04-26  |6800000.0    |6120000.0       |680000.0    |2               |2026-04|
|2026-04-28  |3.48E7       |3.132E7         |3480000.0   |3               |2026-04|
+------------+-------------+----------------+------------+----------------+-------+
only showing top 5 rows


### Cell 2 — Feature Engineering

In [0]:
# ── Feature Engineering cho Time Series ────────────────────
window_ordered = Window.orderBy("booking_date")
window_7d      = window_ordered.rowsBetween(-7, -1)

df_ml = df_rev \
    .withColumn("day_of_week",
                dayofweek("booking_date").cast("double")) \
    .withColumn("month_num",
                month("booking_date").cast("double")) \
    .withColumn("rolling_7d_avg",
                _avg("total_revenue").over(window_7d)) \
    .withColumn("lag_1",
                lag("total_revenue", 1).over(window_ordered)) \
    .withColumn("lag_7",
                lag("total_revenue", 7).over(window_ordered)) \
    .fillna(0) \
    .filter(col("total_revenue") > 0)

FEATURE_COLS_LR = [
    "day_of_week", "month_num",
    "rolling_7d_avg", "lag_1", "lag_7",
    "num_transactions"
]

print(f"-> Feature engineering xong: {df_ml.count()} rows sau fillna")
df_ml.select("booking_date", *FEATURE_COLS_LR, "total_revenue") \
     .show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


-> Feature engineering xong: 32 rows sau fillna
+------------+-----------+---------+--------------------+---------+-----+----------------+-------------+
|booking_date|day_of_week|month_num|rolling_7d_avg      |lag_1    |lag_7|num_transactions|total_revenue|
+------------+-----------+---------+--------------------+---------+-----+----------------+-------------+
|2026-04-22  |4.0        |4.0      |0.0                 |0.0      |0.0  |3               |1.57E7       |
|2026-04-23  |5.0        |4.0      |1.57E7              |1.57E7   |0.0  |6               |3.81E7       |
|2026-04-25  |7.0        |4.0      |2.69E7              |3.81E7   |0.0  |5               |3.43E7       |
|2026-04-26  |1.0        |4.0      |2.9366666666666668E7|3.43E7   |0.0  |2               |6800000.0    |
|2026-04-28  |3.0        |4.0      |2.3725E7            |6800000.0|0.0  |3               |3.48E7       |
+------------+-----------+---------+--------------------+---------+-----+----------------+-------------+
only sh

### Cell 3 — Train/Test Split

In [0]:
# ── Train/Test Split 80/20 (theo thời gian, không random) ──
total    = df_ml.count()
tr_size  = max(int(total * 0.8), 1)
te_size  = total - tr_size

# Lấy ngày phân chia
cutoff_date = df_ml.orderBy("booking_date") \
    .limit(tr_size) \
    .agg({"booking_date": "max"}) \
    .collect()[0][0]

df_train = df_ml.filter(col("booking_date") <= cutoff_date)
df_test  = df_ml.filter(col("booking_date") >  cutoff_date)

print(f"=> Split theo thời gian:")
print(f"   Train: {df_train.count()} ngày (đến {cutoff_date})")
print(f"   Test : {df_test.count()} ngày")

if df_test.count() == 0:
    print("=>  Test set rỗng — dữ liệu quá ít")
    print("   -> Dùng toàn bộ data làm train, evaluate trên train")
    df_test = df_train

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


=> Split theo thời gian:
   Train: 25 ngày (đến 2026-05-21)
   Test : 7 ngày


### Cell 4 — Train Linear Regression

In [0]:
# ── Train Linear Regression ─────────────────────────────────
assembler_lr = VectorAssembler(
    inputCols=FEATURE_COLS_LR,
    outputCol="features",
    handleInvalid="skip"
)
lr = LinearRegression(
    featuresCol="features",
    labelCol="total_revenue",
    maxIter=100,
    regParam=0.1,      # regularization để tránh overfitting với data ít
    elasticNetParam=0.0
)
pipeline_lr = Pipeline(stages=[assembler_lr, lr])

evaluator_rmse = RegressionEvaluator(
    labelCol="total_revenue",
    predictionCol="prediction",
    metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="total_revenue",
    predictionCol="prediction",
    metricName="r2"
)

with mlflow.start_run(run_name="revenue_forecasting_lr"):
    mlflow.log_param("features",       FEATURE_COLS_LR)
    mlflow.log_param("train_days",     df_train.count())
    mlflow.log_param("test_days",      df_test.count())
    mlflow.log_param("regParam",       0.1)
    mlflow.log_param("maxIter",        100)

    model_lr    = pipeline_lr.fit(df_train)
    predictions = model_lr.transform(df_test)

    rmse = evaluator_rmse.evaluate(predictions)
    r2   = evaluator_r2.evaluate(predictions)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2",   r2)

    coefs = model_lr.stages[-1].coefficients
    mlflow.log_param("coefficients", str(coefs.toArray().tolist()))

    print("=" * 55)
    print("=> LINEAR REGRESSION RESULTS — GỬI CHO [A] HÀ")
    print("=" * 55)
    print(f"  RMSE : {rmse:>15,.0f} VNĐ")
    print(f"  R²   : {r2:>15.4f}")
    print("=" * 55)

    if r2 >= 0.7:
        print("  -> Đánh giá: TỐT (R² ≥ 0.7)")
    elif r2 >= 0.3:
        print("  -> Đánh giá: CHẤP NHẬN ĐƯỢC (0.3 ≤ R² < 0.7)")
    else:
        print("  -> Đánh giá: THẤP (R² < 0.3)")
        print("    Slide: 'Dữ liệu ít nên chưa tối ưu,")
        print("    pipeline và approach đúng hướng'")

    print(f"\n  Feature importance (coefficients):")
    for f, c in zip(FEATURE_COLS_LR, coefs):
        print(f"    {f:<20}: {c:>12.2f}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


=> LINEAR REGRESSION RESULTS — GỬI CHO [A] HÀ
  RMSE :       4,516,570 VNĐ
  R²   :          0.8790
  -> Đánh giá: TỐT (R² ≥ 0.7)

  Feature importance (coefficients):
    day_of_week         :    194843.25
    month_num           :  -5839374.31
    rolling_7d_avg      :         0.06
    lag_1               :        -0.09
    lag_7               :         0.02
    num_transactions    :   7894842.01


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Cell 5 — Lưu kết quả

In [0]:
# ── Lưu actual vs predicted ─────────────────────────────────
df_compare = predictions.select(
    "booking_date",
    col("total_revenue").alias("actual"),
    col("prediction").alias("predicted")
).orderBy("booking_date")

df_compare.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_revenue_forecast")

print(f"-> ml_revenue_forecast: {df_compare.count()} records")
print("   Gửi cho [C] Phụng để làm line chart actual vs predicted")

display(df_compare)
# → Chọn Line Chart, X=booking_date
#   Y1=actual (màu xanh), Y2=predicted (màu đỏ)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


-> ml_revenue_forecast: 7 records
   Gửi cho [C] Phụng để làm line chart actual vs predicted


booking_date,actual,predicted
2026-05-22,2.73E7,3.0674103717493273E7
2026-05-23,9100000.0,1.4841881972803578E7
2026-05-24,3.29E7,3.1229542480467845E7
2026-05-25,3.78E7,3.740216598562412E7
2026-05-26,5100000.0,-2105690.0454387106
2026-05-27,7600000.0,8688900.30010739
2026-05-28,6800000.0,289573.53479719535


### Cell 6 — Checklist Day 7

In [0]:
print("=" * 55)
print("=> CHECKLIST NGÀY 7")
print("=" * 55)

items_d7 = {}

try:
    n = spark.read.table("ml_revenue_forecast").count()
    items_d7["ml_revenue_forecast"] = f"--OK-- {n} records"
except:
    items_d7["ml_revenue_forecast"] = "--ERROR-- Chưa có"

try:
    n = spark.read.table("ml_tour_recommendations").count()
    items_d7["ml_tour_recommendations"] = f"--OK-- {n} rows"
except:
    items_d7["ml_tour_recommendations"] = \
        "--WARNING--  Đã tạo ở Day 6 Cell 14"

try:
    items_d7["RMSE"] = f"--OK-- {rmse:,.0f} VNĐ"
    items_d7["R²"]   = f"--OK-- {r2:.4f}"
except:
    items_d7["Metrics"] = "--ERROR Chưa chạy"

try:
    runs = mlflow.search_runs(
        experiment_names=["tourgo-ml-experiments"]
    )
    lr_runs = runs[runs["tags.mlflow.runName"]
                   .str.contains("forecast", case=False, na=False)]
    items_d7["MLflow forecasting run"] = f"--OK-- {len(lr_runs)} run(s)"
except:
    items_d7["MLflow forecasting run"] = "--WARNING--  Kiểm tra thủ công"

for k, v in items_d7.items():
    print(f"  {k:<35} | {v}")

print("=" * 55)
print("-> Gửi RMSE và R² cho [A] Hà đánh giá")
print("-> Gửi ml_revenue_forecast cho [C] Phụng làm chart")

=> CHECKLIST NGÀY 7


2026/06/02 13:51:34 WARNING mlflow.tracking.fluent: Cannot retrieve experiment by name tourgo-ml-experiments


  ml_revenue_forecast                 | --OK-- 7 records
  ml_tour_recommendations             | --OK-- 23 rows
  RMSE                                | --OK-- 4,516,570 VNĐ
  R²                                  | --OK-- 0.8790
  MLflow forecasting run              | --WARNING--  Kiểm tra thủ công
-> Gửi RMSE và R² cho [A] Hà đánh giá
-> Gửi ml_revenue_forecast cho [C] Phụng làm chart
